<img src="http://doe.iitm.ac.in/wp-content/uploads/2021/06/cropped-245-2451831_iit-madras-logo.png" alt="IIT Madras Logo" width="150"/>

**Name:** Ritik Ranjan  
**Roll Number:** [roll no. removed]  
**Course:** BS Data Science & Applications  
**Institution:** IIT Madras  
**Project:** Deep Learning NPPE

## Installing Dependencies

In [1]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch xformers --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes datasets

In [2]:
import os
import numpy as np
import pandas as pd

import torch
from trl import SFTTrainer
from transformers import TrainingArguments, TextStreamer
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import Dataset, load_dataset, DatasetDict

from transformers import AutoModelForSequenceClassification

from transformers import (AutoModelForCausalLM, 
                          DataCollatorForLanguageModeling,
                          AutoTokenizer, 
                          BitsAndBytesConfig, 
                          TrainingArguments,
                          Trainer,
                          DataCollatorForSeq2Seq,
                          logging)
from sklearn.metrics import (accuracy_score, 
                             classification_report, 
                             confusion_matrix)
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Base Model Loading

In [3]:
base_model_name = "/kaggle/input/llama-3.1/transformers/8b-instruct/2"

max_seq_length = 2048
dtype = None 
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = base_model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=32,
    lora_dropout=0.1,
    bias = "none",
    target_modules=["q_proj", "k_proj", "v_proj", "up_proj", "down_proj", "o_proj", "gate_proj"],
    use_rslora=True,
    use_gradient_checkpointing="unsloth",
    loftq_config = None,
    random_state = 3407
)
print(model.print_trainable_parameters())

==((====))==  Unsloth 2025.2.12: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/kaggle/input/llama-3.1/transformers/8b-instruct/2 does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.2.12 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 83,886,080 || all params: 8,114,147,328 || trainable%: 1.0338
None


## Prompt Engineering

In [4]:
data_prompt = """Classify the text into 'Positive', 'Negative', and return the answer as the predicted sentiment.
### Input:
{}
### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompt(examples):
    inputs       = examples["sentence"]
    outputs      = examples["label"]
    texts = []
    for input_, output in zip(inputs, outputs):
        text = data_prompt.format(input_, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

## Loading Train & Test Datasets

In [5]:
train_df = pd.read_csv("/kaggle/input/multi-lingual-sentiment-analysis/train.csv")
test_df = pd.read_csv("/kaggle/input/multi-lingual-sentiment-analysis/test.csv")

In [6]:
training_data = Dataset.from_pandas(train_df)
testing_data = Dataset.from_pandas(test_df)
training_data = training_data.map(formatting_prompt, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Configurations

In [7]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

sft_config = SFTConfig(
    learning_rate = 2e-4,
    dataset_text_field="text",  
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=20,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=1,
    optim="adamw_8bit",
    lr_scheduler_type = "linear",
    weight_decay=0.01,
    warmup_steps=5,
    max_steps=70,
    output_dir="output",
    seed=3407,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=training_data,
    args=sft_config,
)


Applying chat template to train dataset (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

Packing train dataset (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


## Training the Model

In [8]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 101 | Num Epochs = 6
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 70
 "-____-"     Number of trainable parameters = 83,886,080


Step,Training Loss
1,1.412500
2,1.455900
3,1.273400
4,1.032000
5,0.987900
6,1.003000
7,0.895300
8,0.860300
9,0.801200
10,0.891800


TrainOutput(global_step=70, training_loss=0.5377761666263853, metrics={'train_runtime': 4243.3475, 'train_samples_per_second': 0.132, 'train_steps_per_second': 0.016, 'total_flos': 5.315324471672832e+16, 'train_loss': 0.5377761666263853})

In [9]:
model = FastLanguageModel.for_inference(model)

In [10]:
X_test = testing_data.to_pandas()
X_train = training_data.to_pandas()
unique_labels = X_train['label'].unique()
print(unique_labels)

['Positive' 'Negative']


In [11]:
from tqdm import tqdm

def predict(test, model, tokenizer):
    global unique_labels
    y_pred = []
    categories = unique_labels
    
    for i in tqdm(range(len(test))):
        sent = test.iloc[i]["sentence"]
        inputs = tokenizer(
            [data_prompt.format(sent,"",)], return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=10, temperature=0.1)

        answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
        answer = answer.split("### Response:")[-1].strip()

        for category in categories:
            if category.lower() in answer.lower():
                y_pred.append(category)
                break
        else:
            y_pred.append("Positive")
    
    return y_pred

y_pred = predict(X_test, model, tokenizer)

100%|██████████| 100/100 [00:53<00:00,  1.88it/s]


## Creating Submission File

In [12]:
submission = pd.DataFrame({
    "ID": range(1,101),
    "label": y_pred
})
submission.to_csv("submission.csv", index = False)

## Checking Label Counts

In [13]:
check_none = pd.read_csv("/kaggle/working/submission.csv")
print(check_none['label'].unique())

['Positive' 'Negative']


In [14]:
check_none['label'].value_counts()

label
Negative    53
Positive    47
Name: count, dtype: int64